# CMBS Loan-Level Data Cleaning Pipeline
**Author:** Ruoling (Iris) Li 
**Description:** A documented, reproducible pipeline for cleaning raw Trepp CMBS loan-level data

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# ── Load raw data ──────────────────────────────────────────────────────────────
RAW_PATH = 'prop_CA_2019_2021.csv'

df_raw = pd.read_csv(RAW_PATH, low_memory=False, encoding='latin1')
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Memory (raw): {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Loaded: 105,338 rows × 323 columns
Memory (raw): 567.3 MB


## 2. Drop Sparse Columns

In [2]:
# ── Missingness audit ─────────────────────────────────────────────────────────
valid_pct = df_raw.notna().mean() * 100   # % non-missing per column

print("Missingness distribution:")
for threshold in [25, 50, 75, 90]:
    n_keep = (valid_pct >= threshold).sum()
    print(f"  ≥{threshold}% non-missing: {n_keep} columns kept ({df_raw.shape[1] - n_keep} dropped)")

# Apply 50% threshold (adjustable)
MISSING_THRESHOLD = 50
cols_to_keep = valid_pct[valid_pct >= MISSING_THRESHOLD].index
df = df_raw.loc[:, cols_to_keep].copy()

print(f"\nKept {len(cols_to_keep)} / {df_raw.shape[1]} columns (threshold = {MISSING_THRESHOLD}%)")
print(f"Memory after triage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Missingness distribution:
  ≥25% non-missing: 141 columns kept (182 dropped)
  ≥50% non-missing: 85 columns kept (238 dropped)
  ≥75% non-missing: 53 columns kept (270 dropped)
  ≥90% non-missing: 51 columns kept (272 dropped)

Kept 85 / 323 columns (threshold = 50%)
Memory after triage: 223.2 MB


## 3. Type Coercion

Raw CSV files store everything as strings. We coerce each column to its correct type:
- Dates parsed with `errors='coerce'` so malformed entries become `NaT` (not silently wrong)
- Numerics parsed with `errors='coerce'` so non-numeric strings become `NaN`
- Unrealistic values (e.g. negative square footage) are nulled out explicitly

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
def to_datetime_safe(series):
    """Parse dates; malformed values → NaT."""
    return pd.to_datetime(series, errors='coerce')

def to_numeric_safe(series):
    """Parse numerics; non-numeric strings → NaN."""
    return pd.to_numeric(series, errors='coerce')

def months_between(d1, d2):
    """Fractional months from d1 → d2. Returns NaN for missing dates."""
    d1 = pd.to_datetime(d1, errors='coerce')
    d2 = pd.to_datetime(d2, errors='coerce')
    return (d2 - d1).dt.days / 30.4375

def safe_div(a, b):
    """Division that returns NaN (not inf) when denominator is 0 or missing."""
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    return (a / b).replace([np.inf, -np.inf], np.nan)

# ── Date columns ───────────────────────────────────────────────────────────────
date_cols = [
    'distdate', 'occdate', 'leasereviewdate', 'inspectiondate',
    'lastpropcontribdate', 'curlessee1expdt',
    'leaseExpDt1', 'leaseExpDt2', 'leaseExpDt3', 'securappvaluedate'
]
for col in date_cols:
    if col in df.columns:
        before_nulls = df[col].isna().sum()
        df[col] = to_datetime_safe(df[col])
        after_nulls  = df[col].isna().sum()
        new_nulls = after_nulls - before_nulls
        if new_nulls > 0:
            print(f"  {col}: {new_nulls} unparseable values → NaT")

# ── Numeric columns ────────────────────────────────────────────────────────────
num_cols = [
    'rentarea', 'curRentableArea',
    'securrevenues', 'securexpenses', 'securnoi', 'securnetcashflow',
    'secPropBal', 'secLTV',
    'lesseepct1', 'lesseepct2', 'lesseepct3',
    'lesseesqft1', 'lesseesqft2', 'lesseesqft3',
    'lesseepctcurr1', 'occrate', 'securocc', 'propyear'
]
for col in num_cols:
    if col in df.columns:
        df[col] = to_numeric_safe(df[col])


df['sqft_base'] = df.get('curRentableArea', df.get('rentarea'))
if 'rentarea' in df.columns:
    df['sqft_base'] = df['sqft_base'].fillna(df['rentarea'])
df.loc[df['sqft_base'] <= 0, 'sqft_base'] = np.nan

n_bad_sqft = (df['sqft_base'] <= 0).sum()
print(f"\nsqft_base: {n_bad_sqft} non-positive values set to NaN")
print(f"sqft_base range after cleaning: [{df['sqft_base'].min():,.0f}, {df['sqft_base'].max():,.0f}]")


sqft_base: 0 non-positive values set to NaN
sqft_base range after cleaning: [1,624, 2,074,270]


## 4. Address Normalization & Geocode Merge

Property addresses in raw CMBS data are inconsistently formatted (mixed case, varying abbreviations, extra punctuation). We normalize to a canonical form before merging geocoded coordinates from an ArcGIS export.

In [4]:
# ── Address normalizer ────────────────────────────────────────────────────────
def norm_addr(s):
    """
    Standardize address strings for fuzzy-free exact matching:
    uppercase → strip punctuation → collapse whitespace → expand abbreviations.
    """
    if pd.isna(s):
        return ''
    s = str(s).upper().strip()
    s = re.sub(r'[^\w\s]', ' ', s)   # remove punctuation
    s = re.sub(r'\s+', ' ', s)        # collapse whitespace
    abbrevs = {
        ' STREET': ' ST', ' AVENUE': ' AVE', ' BOULEVARD': ' BLVD',
        ' ROAD': ' RD', ' DRIVE': ' DR', ' LANE': ' LN',
        ' SUITE': ' STE', ' NORTH': ' N', ' SOUTH': ' S',
        ' EAST': ' E', ' WEST': ' W',
    }
    for full, short in abbrevs.items():
        s = s.replace(full, short)
    return s.strip()

# ── Build join key in CMBS data ───────────────────────────────────────────────
# Combine address + city + state + first-5-digits of zip for precision
for col in ['address', 'city', 'state', 'zip']:
    if col not in df.columns:
        df[col] = np.nan

df['addr_key'] = (
    df['address'].apply(norm_addr) + ' | ' +
    df['city'].apply(norm_addr)    + ' | ' +
    df['state'].apply(norm_addr)   + ' | ' +
    df['zip'].astype(str).str[:5].apply(norm_addr)
)

# ── Load ArcGIS geocoded file & build matching key ────────────────────────────
ARCGIS_PATH = 'data/unique_retail_addresses_arcgis.csv'

if os.path.exists(ARCGIS_PATH):
    arc = pd.read_csv(ARCGIS_PATH, encoding='utf-8-sig')

    # Filter to high-confidence geocodes only (score ≥ 85 out of 100)
    arc_clean = arc[arc['Score'] >= 85].copy()
    print(f"ArcGIS: {len(arc):,} total → {len(arc_clean):,} high-confidence (Score ≥ 85)")

    arc_clean['addr_key'] = arc_clean['Match_addr'].apply(norm_addr)

    # ── Left-join coordinates onto CMBS data ─────────────────────────────────
    df = df.merge(
        arc_clean[['addr_key', 'X', 'Y', 'Score']].rename(columns={'X': 'lon_arc', 'Y': 'lat_arc'}),
        on='addr_key', how='left'
    )

    # Fill existing coordinate columns or create new ones
    df['long'] = df.get('long', pd.Series(np.nan, index=df.index)).fillna(df['lon_arc'])
    df['lat']  = df.get('lat',  pd.Series(np.nan, index=df.index)).fillna(df['lat_arc'])

    matched = df['lat'].notna().sum()
    print(f"Geocode match rate: {matched:,} / {len(df):,} ({matched/len(df)*100:.1f}%)")
else:
    print(f"ArcGIS file not found at {ARCGIS_PATH} — skipping geocode merge")
    print("(Expected columns: Match_addr, X, Y, Score)")

ArcGIS file not found at data/unique_retail_addresses_arcgis.csv — skipping geocode merge
(Expected columns: Match_addr, X, Y, Score)


## 5. Feature Engineering

We derive analysis-ready features from the cleaned raw fields. All divisions go through `safe_div` to prevent infinity propagation. Percent columns that arrive on a 0–100 scale are rescaled to 0–1.

### 5a. Income-Side Features (per sqft)

In [5]:
# Revenue, expense, NOI, NCF — normalized by rentable sqft
df['revenue_psf'] = safe_div(df.get('securrevenues'), df['sqft_base'])
df['expense_psf'] = safe_div(df.get('securexpenses'), df['sqft_base'])
df['noi_psf']     = safe_div(df.get('securnoi'),      df['sqft_base'])
df['ncf_psf']     = safe_div(df.get('securnetcashflow'), df['sqft_base'])

# Operating ratios
df['expense_ratio'] = safe_div(df.get('securexpenses'), df.get('securrevenues'))
df['noi_margin']    = safe_div(df.get('securnoi'),      df.get('securrevenues'))
df['ncf_margin']    = safe_div(df.get('securnetcashflow'), df.get('securrevenues'))

# Winsorize at 1st/99th percentile to reduce extreme outlier influence
psf_cols = ['revenue_psf', 'expense_psf', 'noi_psf', 'ncf_psf',
            'expense_ratio', 'noi_margin', 'ncf_margin']
for col in psf_cols:
    if col in df.columns:
        lo, hi = df[col].quantile([0.01, 0.99])
        n_clipped = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lo, hi)
        print(f"  {col}: winsorized {n_clipped} outliers [{lo:.3f}, {hi:.3f}]")

  revenue_psf: winsorized 1528 outliers [5.750, 104.674]
  expense_psf: winsorized 1538 outliers [0.156, 27.720]
  noi_psf: winsorized 1893 outliers [5.039, 88.114]
  ncf_psf: winsorized 2005 outliers [4.526, 79.938]
  expense_ratio: winsorized 1542 outliers [0.016, 0.529]
  noi_margin: winsorized 1316 outliers [0.522, 1.000]
  ncf_margin: winsorized 1544 outliers [0.489, 0.997]


### 5b. Occupancy

In [6]:
# Prefer 'securocc' (Trepp-calculated); fall back to 'occrate'
occ_src = 'securocc' if 'securocc' in df.columns else 'occrate'
df['occ_base'] = df[occ_src].copy()

# Rescale 0–100 to 0–1 if needed
if df['occ_base'].median(skipna=True) > 1:
    df['occ_base'] = df['occ_base'] / 100.0
    print(f"Rescaled {occ_src} from 0-100 to 0-1 scale")

df['low_occ_flag']     = (df['occ_base'] < 0.85).astype(float)
df['occ_missing_flag'] = df['occ_base'].isna().astype(float)
print(f"Low occupancy (<85%): {df['low_occ_flag'].sum():,.0f} properties ({df['low_occ_flag'].mean()*100:.1f}%)")

Rescaled securocc from 0-100 to 0-1 scale
Low occupancy (<85%): 4,902 properties (4.7%)


### 5c. Tenant Concentration & Lease Rollover (Top-3 Tenants)

In [7]:
# Rescale lessee pct columns if on 0-100 scale
for col in ['lesseepct1', 'lesseepct2', 'lesseepct3', 'lesseepctcurr1']:
    if col in df.columns:
        if df[col].median(skipna=True) > 1:
            df[col] = df[col] / 100.0

# Months-to-expiration and rollover flags for top 3 tenants
for k in [1, 2, 3]:
    exp_col = f'leaseExpDt{k}'
    if exp_col in df.columns:
        df[f'mte_{k}'] = months_between(df['distdate'], df[exp_col])
    else:
        df[f'mte_{k}'] = np.nan
    df[f'exp_{k}_12m'] = df[f'mte_{k}'].between(0, 12).astype(float)
    df[f'exp_{k}_24m'] = df[f'mte_{k}'].between(0, 24).astype(float)

# Weighted rollover exposure (pct of rent expiring in 12/24 months)
df['pct_top3_sum']     = 0.0
df['pct_exp_12m_top3'] = 0.0
df['pct_exp_24m_top3'] = 0.0

for k in [1, 2, 3]:
    pct = df.get(f'lesseepct{k}', pd.Series(0, index=df.index)).fillna(0)
    df['pct_top3_sum']     += pct
    df['pct_exp_12m_top3'] += pct * df[f'exp_{k}_12m'].fillna(0)
    df['pct_exp_24m_top3'] += pct * df[f'exp_{k}_24m'].fillna(0)

# WAULT: weighted-average unexpired lease term (top 3)
w_num = sum(
    df.get(f'lesseepct{k}', pd.Series(0, index=df.index)).fillna(0) * df[f'mte_{k}']
    for k in [1, 2, 3]
)
df['wault_top3_months'] = safe_div(w_num, df['pct_top3_sum'].replace(0, np.nan))

# HHI concentration index (top 3)
df['top1_share'] = df.get('lesseepct1', np.nan)
df['top3_share'] = sum(df.get(f'lesseepct{k}', pd.Series(0, index=df.index)).fillna(0) for k in [1, 2, 3])
df['hhi_top3']   = sum(df.get(f'lesseepct{k}', pd.Series(0, index=df.index)).fillna(0)**2 for k in [1, 2, 3])

print("Tenant/lease rollover features created.")
print(f"  Mean pct_exp_12m_top3: {df['pct_exp_12m_top3'].mean():.3f}")
print(f"  Mean wault_top3_months: {df['wault_top3_months'].mean():.1f} months")

Tenant/lease rollover features created.
  Mean pct_exp_12m_top3: 0.302
  Mean wault_top3_months: -0.0 months


### 5d. Loan Structure & Interaction Terms

In [8]:
# Property age
if 'propyear' in df.columns and 'distdate' in df.columns:
    df['property_age'] = df['distdate'].dt.year - df['propyear']
    # Null out implausible ages
    df.loc[(df['property_age'] < 0) | (df['property_age'] > 200), 'property_age'] = np.nan

# LTV flags and interactions
df['high_ltv_flag'] = (df.get('secLTV', pd.Series(np.nan, index=df.index)) > 0.75).astype(float)
df['ltv_x_roll12']  = df.get('secLTV', pd.Series(np.nan, index=df.index)) * df['pct_exp_12m_top3']
df['bal_psf']       = safe_div(df.get('secPropBal'), df['sqft_base'])

print("Loan structure features created.")

Loan structure features created.


## 6. Memory Optimization


In [9]:
def optimize_dtypes(df, label_col='securdscr', target_mb=30.0):
    """
    Reduce DataFrame memory footprint by:
      1. Dropping high-cardinality ID / free-text columns
      2. Converting datetime columns to integer days
      3. Downcasting float64 → float32, int64 → int32
      4. Converting low-cardinality object columns to category
      5. Dropping columns with >80% missingness (if still above target)
    """
    def mem_mb(x):
        return x.memory_usage(deep=True).sum() / 1e6

    df2 = df.copy()
    print(f"Start:  {mem_mb(df2):.1f} MB | {df2.shape[1]} columns")

    # 1. Drop high-memory string columns not useful as ML features
    drop_candidates = [
        'dosname', 'transidtrustee', 'poolnum', 'collateralid', 'servicerpropid',
        'trepppropnum', 'masterloanidtrepp', 'notenum',
        'propname', 'address', 'zip',
        'lessee1', 'lessee2', 'lessee3', 'curlessee1',
        'city', 'county', 'submarket',
    ]
    df2 = df2.drop(columns=[c for c in drop_candidates if c in df2.columns])
    print(f"After drop strings:   {mem_mb(df2):.1f} MB")

    # 2. Convert datetime → integer days since epoch (safe for NaN via Int32)
    date_cols = [c for c in df2.columns if np.issubdtype(df2[c].dtype, np.datetime64)]
    for c in date_cols:
        df2[c + '_days'] = (df2[c].view('int64') // 86_400_000_000_000).astype('Int32')
    df2 = df2.drop(columns=date_cols)
    print(f"After datetime→days:  {mem_mb(df2):.1f} MB")

    # 3. Downcast numerics
    for c in df2.select_dtypes('float64').columns:
        df2[c] = pd.to_numeric(df2[c], downcast='float')
    for c in df2.select_dtypes(['int64', 'int32']).columns:
        df2[c] = pd.to_numeric(df2[c], downcast='integer')
    print(f"After numeric cast:   {mem_mb(df2):.1f} MB")

    # 4. Low-cardinality strings → category
    for c in df2.select_dtypes('object').columns:
        if df2[c].nunique(dropna=True) <= 200:
            df2[c] = df2[c].astype('category')
    print(f"After obj→category:   {mem_mb(df2):.1f} MB")

    # 5. Drop columns with >80% missingness if still over target
    if mem_mb(df2) > target_mb:
        miss = df2.isna().mean()
        drop_sparse = miss[miss > 0.80].index.difference([label_col]).tolist()
        df2 = df2.drop(columns=drop_sparse)
        print(f"After drop >80% miss: {mem_mb(df2):.1f} MB ({len(drop_sparse)} cols dropped)")

    print(f"Final:  {mem_mb(df2):.1f} MB | {df2.shape[1]} columns")
    return df2

df_clean = optimize_dtypes(df, label_col='securdscr', target_mb=30.0)

Start:  259.6 MB | 117 columns
After drop strings:   162.3 MB
After datetime→days:  159.2 MB
After numeric cast:   130.3 MB
After obj→category:   56.9 MB
After drop >80% miss: 56.4 MB (1 cols dropped)
Final:  56.4 MB | 98 columns


## 8. Quick Sanity Check — Does Cleaned Data Predict Risk?

Before handing off to full modeling, we run a lightweight cross-validated check to confirm the engineered features carry real signal. This is not a final model: if nothing predicts at all, something upstream is wrong.

We use a 5-fold cross-validation on two models (Logistic Regression and Random Forest) and report ROC-AUC. A score meaningfully above 0.5 confirms the cleaning pipeline preserved the economic structure of the data.

In [12]:
# ── Define at-risk label ──────────────────────────────────────────────────────
# DSCR < 1.2 is the standard servicer watchlist threshold
TARGET = 'securdscr'
if TARGET not in df_clean.columns:
    print("Target column not found — skipping ML check")
else:
    df_ml = df_clean[df_clean[TARGET].notna()].copy()
    df_ml['at_risk'] = (df_ml[TARGET] < 1.2).astype(int)

    print(f"At-risk rate: {df_ml['at_risk'].mean()*100:.1f}%  "
          f"({df_ml['at_risk'].sum():,} / {len(df_ml):,} loans)")

    # ── Feature set: only engineered columns (no raw DSCR components) ─────────
    feature_cols = [c for c in [
        'sqft_base', 'secLTV', 'occ_base', 'low_occ_flag',
        'revenue_psf', 'expense_psf', 'noi_psf', 'noi_margin', 'expense_ratio',
        'pct_exp_12m_top3', 'wault_top3_months', 'hhi_top3', 'top3_share',
        'high_ltv_flag', 'ltv_x_roll12', 'bal_psf',
    ] if c in df_ml.columns]

    X = df_ml[feature_cols].fillna(df_ml[feature_cols].median())
    y = df_ml['at_risk']

    # ── Logistic Regression ────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_auc = cross_val_score(lr, X_sc, y, cv=5, scoring='roc_auc').mean()

    # ── Random Forest ──────────────────────────────────────────────────────────
    rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                                class_weight='balanced', random_state=42, n_jobs=-1)
    rf_auc = cross_val_score(rf, X, y, cv=5, scoring='roc_auc').mean()

    print(f"\n5-Fold CV ROC-AUC (baseline = 0.50):")
    print(f"  Logistic Regression : {lr_auc:.3f}")
    print(f"  Random Forest       : {rf_auc:.3f}")

    # ── Feature importance (RF) — quick signal check ───────────────────────────
    rf.fit(X, y)
    importances = sorted(zip(feature_cols, rf.feature_importances_),
                         key=lambda x: x[1], reverse=True)

    print(f"\nTop 5 features by RF importance:")
    for feat, imp in importances[:5]:
        bar = '█' * int(imp * 200)
        print(f"  {feat:<25} {imp:.4f}  {bar}")


At-risk rate: 1.4%  (779 / 54,129 loans)

5-Fold CV ROC-AUC (baseline = 0.50):
  Logistic Regression : 0.874
  Random Forest       : 0.999

Top 5 features by RF importance:
  secLTV                    0.2309  ██████████████████████████████████████████████
  ltv_x_roll12              0.1208  ████████████████████████
  noi_psf                   0.0834  ████████████████
  expense_ratio             0.0804  ████████████████
  bal_psf                   0.0735  ██████████████


## 7. Validation & Output Summary

In [11]:
# ── Final completeness report ──────────────────────────────────────────────────
key_features = [
    'sqft_base', 'secLTV', 'occ_base',
    'revenue_psf', 'noi_psf', 'noi_margin',
    'pct_exp_12m_top3', 'wault_top3_months', 'hhi_top3',
    'lat', 'long', 'securdscr'
]

print("Key feature completeness:")
print(f"  {'Column':<25} {'Non-null':>10} {'Complete %':>12}")
print("  " + "-" * 50)
for col in key_features:
    if col in df_clean.columns:
        n = df_clean[col].notna().sum()
        pct = n / len(df_clean) * 100
        print(f"  {col:<25} {n:>10,} {pct:>11.1f}%")

print(f"\nFinal dataset: {len(df_clean):,} rows × {df_clean.shape[1]} columns")
if 'masterpropidtrepp' in df_clean.columns:
    print(f"Unique properties: {df_clean['masterpropidtrepp'].nunique():,}")
if 'securdscr' in df_clean.columns:
    tgt = df_clean['securdscr'].dropna()
    print(f"\nDSCR (target) — n={len(tgt):,} | mean={tgt.mean():.3f} | "
          f"std={tgt.std():.3f} | at-risk (<1.2): {(tgt<1.2).mean()*100:.1f}%")

# ── Save ───────────────────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
OUT_PATH = 'data/cmbs_clean.csv'
df_clean.to_csv(OUT_PATH, index=False)
size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f"\nSaved: {OUT_PATH} ({size_mb:.1f} MB)")

Key feature completeness:
  Column                      Non-null   Complete %
  --------------------------------------------------
  sqft_base                    105,107        99.8%
  secLTV                       103,643        98.4%
  occ_base                     101,088        96.0%
  revenue_psf                   78,882        74.9%
  noi_psf                       97,223        92.3%
  noi_margin                    76,985        73.1%
  pct_exp_12m_top3             105,338       100.0%
  wault_top3_months             69,794        66.3%
  hhi_top3                     105,338       100.0%
  lat                          104,955        99.6%
  long                         104,955        99.6%
  securdscr                     54,129        51.4%

Final dataset: 105,338 rows × 98 columns
Unique properties: 4,141

DSCR (target) — n=54,129 | mean=1.899 | std=0.655 | at-risk (<1.2): 1.4%

Saved: data/cmbs_clean.csv (68.5 MB)
